# Explainable Career Decision Support System

**Final project — CS 4580/5580 Automated Decision Systems**  
Kevin Rusagara Iraguha

This notebook walks through an automated decision system that recommends
career paths from a structured user profile. Unlike a black-box predictor,
every recommendation comes with a **decision trace**: which rules fired,
which profile attributes triggered them, and how each one contributed to
the score. Four explanation modes make the reasoning fully inspectable:

1. **Top reasons** — positive and negative factors per career
2. **Head-to-head** — what differentiates #1 from #2
3. **Counterfactuals** — minimal profile changes that flip the recommendation
4. **Tradeoffs** — rules that pull in opposing directions

Plus an **option-generation** layer that surfaces hybrid roles, dark-horse
options, and weak-match warnings.

## 1. System overview

In [1]:
from advisor import (
    CAREERS, RULES, UserProfile, score,
    explain_top, head_to_head, counterfactuals, detect_tradeoffs, alternatives,
)
from advisor.scoring import ranked
from sample_profiles import (
    swe_candidate, research_candidate, consulting_candidate,
    conflicted_candidate, early_career_candidate,
)

print(f'Career options in catalog: {len(CAREERS)}')
print(f'Rules in rule base:        {len(RULES)}')
print()
print('Careers:')
for c in CAREERS:
    print(f'  - {c.name}: {c.description}')

Career options in catalog: 13
Rules in rule base:        64

Careers:
  - Software Engineering: Designing and building software products and systems.
  - Data Science: Extracting insights and predictions from data using statistics and ML.
  - Machine Learning Engineering: Building production ML systems, pipelines, and infrastructure.
  - Product Management: Owning product strategy, prioritization, and cross-functional execution.
  - UX / Product Design: Researching users and designing interfaces and experiences.
  - Management Consulting: Advising organizations on strategy, operations, and transformation.
  - Quantitative Finance: Quantitative research, trading, or risk in financial markets.
  - Industry Research: Applied research at scale in industry R&D labs.
  - Academic Research: PhD-track research, publications, and faculty paths.
  - DevOps / SRE: Reliability, infrastructure, and developer-platform engineering.
  - Cybersecurity: Securing systems — defensive, offensive, or govern

## 1b. Rule base — what's inside

The 64 rules fall into seven thematic blocks. The cell below groups them
by category and prints counts so you can see the shape of the rule base
before walking through individual decisions.


In [2]:
from collections import Counter

def rule_category(rid):
    if rid.startswith('interest_'):  return 'Interest'
    if rid.startswith('style_'):     return 'Work style'
    if rid.startswith('exp_'):       return 'Experience'
    if rid.startswith('goal_'):      return 'Goal'
    if rid.startswith(('lacks_', 'constraint_')): return 'Constraint'
    if '_plus_' in rid or 'theoretical' in rid:   return 'Compound'
    return 'Skill'

buckets = {}
for r in RULES:
    buckets.setdefault(rule_category(r.id), []).append(r)

for cat in ['Skill', 'Interest', 'Work style', 'Experience', 'Goal',
            'Constraint', 'Compound']:
    rules_in = buckets.get(cat, [])
    print(f'{cat:15s} ({len(rules_in):2d} rules)')
    for r in rules_in:
        print(f'    - {r.id:30s} {r.description}')
    print()


Skill           (17 rules)
    - strong_programming             Strong general-purpose programming supports engineering roles.
    - ml_skills                      Hands-on machine learning experience is core to ML/DS roles.
    - stats_math                     Strong statistics / math underpins data and quantitative work.
    - systems_skills                 Systems / infrastructure skills support reliability and security work.
    - strong_communication           Strong communication is critical for client-facing and cross-functional roles.
    - strong_writing                 Strong writing is leveraged in research, PM specs, and consulting decks.
    - design_skills                  Visual / interaction design skills are central to UX and helpful for PM.
    - leadership                     Demonstrated leadership supports management-track roles.
    - finance_skills                 Finance / market knowledge supports quantitative finance roles.
    - security_skills               

## 2. Profile A — clear software-engineering candidate

Strong programming skills, prior SWE internship, building / shipping interests,
and a long-term goal of starting a company. We'd expect SWE and entrepreneurship
to dominate, with the goal of starting a company tilting things toward founder paths.

In [3]:
profile = swe_candidate()
print(profile.summary())

Profile: Alex (SWE-leaning senior)
  Technical: python(5), java(4), javascript(4), system_design(4), linux(4)
  Non-technical: communication(3), writing(3), leadership(2)
  Interests: building, products, ai
  Work style: hands_on, collaborative, fast_paced
  Short-term goals: land a high-impact engineering role at a product company; earn a competitive starting salary
  Long-term goals: become a tech lead and eventually start a company


In [4]:
scores, firings = score(profile)
print(explain_top(scores, firings, top_n=3))

TOP RECOMMENDATIONS

#1  Machine Learning Engineering   (score: +11.27)
    Why this ranks high:
      +2.00  Strong general-purpose programming supports engineering roles.
             - python: 5/5
             - java: 4/5
             - javascript: 4/5
      +2.00  Interest in AI/ML drives toward ML, DS, and research paths.
             - interest: ai
      +1.80  Hands-on machine learning experience is core to ML/DS roles.
             - machine learning: 3/5
      +1.50  Hands-on builders prefer engineering-track roles.
             - work style: hands_on
      +1.50  ML expertise combined with systems skills is exactly the ML-engineering profile.
             - machine learning: 3/5
             - system design: 4/5

#2  Entrepreneurship   (score: +10.50)
    Why this ranks high:
      +4.00  Wanting to start a company favors the founder path.
             - goal mentions 'start a company'
      +2.50  Fast-paced preference suits startups and consulting sprints.
             - wo

In [5]:
print(alternatives(profile, scores, firings))

ALTERNATIVE OPTIONS
  CLOSE CALL: Machine Learning Engineering (+11.27) and Entrepreneurship (+10.50) are very close.
     Consider: Founding ML engineer at an early-stage AI startup
  DARK HORSE: Product Management has substantial positive support (+6.00 of positive evidence) but is held back by penalties. Worth considering if those constraints can be addressed.


In [6]:
top_two = ranked(scores)[:2]
print(head_to_head(top_two[0][0], top_two[1][0], firings))

HEAD-TO-HEAD: Machine Learning Engineering  vs  Entrepreneurship
  Δ -4.00  (+0.00 vs +4.00)  →  favors Entrepreneurship
             Wanting to start a company favors the founder path.
  Δ -2.50  (+0.00 vs +2.50)  →  favors Entrepreneurship
             Fast-paced preference suits startups and consulting sprints.
  Δ +2.00  (+2.00 vs +0.00)  →  favors Machine Learning Engineering
             Interest in AI/ML drives toward ML, DS, and research paths.
  Δ +1.80  (+1.80 vs +0.00)  →  favors Machine Learning Engineering
             Hands-on machine learning experience is core to ML/DS roles.
  Δ +1.50  (+2.00 vs +0.50)  →  favors Machine Learning Engineering
             Strong general-purpose programming supports engineering roles.
  Δ +1.50  (+1.50 vs +0.00)  →  favors Machine Learning Engineering
             Hands-on builders prefer engineering-track roles.
  Δ -1.50  (+0.00 vs +1.50)  →  favors Entrepreneurship
             Wanting impact / mission alignment supports research and 

In [7]:
print(counterfactuals(profile, scores))

COUNTERFACTUALS — what would change the recommendation?
  If you raised design visual to 4/5, top recommendation would shift to Entrepreneurship.
  If you raised communication to 4/5, top recommendation would shift to Entrepreneurship.
  If you raised leadership to 4/5, top recommendation would shift to Entrepreneurship.


**What to notice.** The decision trace shows *exactly* which rules fired and which
profile attributes triggered them. The head-to-head explains the small gap between
the top two careers as a tradeoff between founder-direction goals and concrete SWE
evidence. The counterfactual is the system's way of saying: *here is a small
profile change that would change my mind.*

## 3. Profile B — research-track candidate

Deep ML / math background, research experience, an explicit PhD goal. The system
should land decisively on academic research, with industry research as a strong
second.

In [8]:
profile = research_candidate()
print(profile.summary())
scores, firings = score(profile)
print()
print(explain_top(scores, firings, top_n=3))

Profile: Priya (research-track)
  Technical: machine_learning(5), deep_learning(5), math(5), statistics(5), python(4)
  Non-technical: writing(5), communication(3)
  Interests: research, ai, open_problems, theory
  Work style: independent, strategic
  Short-term goals: apply to top PhD programs in machine learning
  Long-term goals: pursue a PhD and contribute to fundamental AI research

TOP RECOMMENDATIONS

#1  Academic Research   (score: +19.10)
    Why this ranks high:
      +4.00  Wanting a PhD strongly favors the academic track.
             - goal mentions 'phd'
      +3.00  Curiosity about open problems fits research-oriented careers.
             - interest: research
             - interest: open_problems
             - interest: theory
      +2.00  Strong statistics / math underpins data and quantitative work.
             - statistics: 5/5
             - math: 5/5
             - linear algebra: 4/5
      +2.00  Strong writing is leveraged in research, PM specs, and consulting

In [9]:
print(counterfactuals(profile, scores))

COUNTERFACTUALS — what would change the recommendation?
  The top recommendation is robust — no single small change to the profile would flip it.


**What to notice.** When the top recommendation is well-supported by many
independent rules, the counterfactual search reports that no small profile
change would flip it — i.e., the recommendation is *robust*. That's an
important property of an explainable system.

## 4. Profile C — consulting-aspiring candidate (with a constraint twist)

Strong communication and strategic thinking, an explicit consulting goal —
**but** a `no_relocation` constraint. Consulting jobs are concentrated in major
cities, so the constraint penalizes consulting. We'd expect the system to surface
this as a tradeoff.

In [10]:
profile = consulting_candidate()
print(profile.summary())
scores, firings = score(profile)
print()
print(explain_top(scores, firings, top_n=3))

Profile: Jordan (strategy / consulting)
  Technical: math(3), statistics(3), python(2)
  Non-technical: communication(5), presentation(5), writing(4), leadership(4)
  Interests: strategy, business, users
  Work style: collaborative, fast_paced, strategic, ambiguous
  Short-term goals: land an offer at a top management consulting firm
  Long-term goals: move into product strategy or general management
  Constraints: no_relocation

TOP RECOMMENDATIONS

#1  Product Management   (score: +16.20)
    Why this ranks high:
      +3.00  Strong communication is critical for client-facing and cross-functional roles.
             - communication: 5/5
             - presentation: 5/5
      +2.50  Strategic thinkers gravitate to product and consulting roles.
             - work style: strategic
      +2.00  Interest in strategy fits consulting and product roles.
             - interest: strategy
             - interest: business
      +2.00  Interest in users / human factors supports UX and PM.
    

In [11]:
top_two = ranked(scores)[:2]
print(head_to_head(top_two[0][0], top_two[1][0], firings))

HEAD-TO-HEAD: Product Management  vs  Management Consulting
  Δ +2.00  (+2.00 vs +0.00)  →  favors Product Management
             Interest in users / human factors supports UX and PM.
  Δ +2.00  (+0.00 vs -2.00)  →  favors Product Management
             Inability to relocate cuts options concentrated in specific cities.
  Δ -1.00  (+2.00 vs +3.00)  →  favors Management Consulting
             Interest in strategy fits consulting and product roles.
  Δ -1.00  (+1.00 vs +2.00)  →  favors Management Consulting
             Comfort with ambiguity is a strong fit for founder and consultant roles.
  Δ -0.50  (+1.50 vs +2.00)  →  favors Management Consulting
             Collaborative preference suits cross-functional roles.
  Δ -0.50  (+1.00 vs +1.50)  →  favors Management Consulting
             Fast-paced preference suits startups and consulting sprints.
  Δ +0.50  (+2.50 vs +2.00)  →  favors Product Management
             Strategic thinkers gravitate to product and consulting roles.
  

In [12]:
print(counterfactuals(profile, scores))

COUNTERFACTUALS — what would change the recommendation?
  If the constraint 'no_relocation' were removed, top recommendation would shift to Management Consulting.


**What to notice.** The user *said* they wanted consulting, but the system
reports that PM ranks higher *because* of the relocation constraint. That's
exactly the kind of insight a black-box recommender would hide. The
counterfactual then says: *if you removed the no_relocation constraint, the
recommendation would shift back to consulting.* The user gets the full picture
and can decide for themselves whether to relax the constraint.

## 5. Profile D — conflicted candidate (skills vs. preferences)

Strong programming background plus solid communication and leadership. The user
wants a role combining technical depth with product impact. This is the kind of
profile where the alternative-options engine adds the most value.

In [13]:
profile = conflicted_candidate()
print(profile.summary())
scores, firings = score(profile)
print()
print(explain_top(scores, firings, top_n=3))

Profile: Sam (conflicted: SWE skills, PM aspirations)
  Technical: python(5), java(4), system_design(3), machine_learning(3)
  Non-technical: communication(4), presentation(4), leadership(4), writing(3)
  Interests: building, products, strategy, users
  Work style: collaborative, strategic
  Short-term goals: find a role that combines technical depth with product impact
  Long-term goals: become a product leader at a major tech company

TOP RECOMMENDATIONS

#1  Product Management   (score: +15.30)
    Why this ranks high:
      +2.50  Strategic thinkers gravitate to product and consulting roles.
             - work style: strategic
      +2.40  Strong communication is critical for client-facing and cross-functional roles.
             - communication: 4/5
             - presentation: 4/5
      +2.00  Interest in strategy fits consulting and product roles.
             - interest: strategy
      +2.00  Interest in users / human factors supports UX and PM.
             - interest: users


In [14]:
print(alternatives(profile, scores, firings))

ALTERNATIVE OPTIONS
  DARK HORSE: Technical Program Management has substantial positive support (+9.90 of positive evidence) but is held back by penalties. Worth considering if those constraints can be addressed.


## 6. Profile E — early-career, weak signal

Few skills, no experience, no strong goals. We expect every score to be low —
the system should flag that the profile doesn't yet provide enough signal to
make a confident recommendation.

In [15]:
profile = early_career_candidate()
print(profile.summary())
scores, firings = score(profile)
print()
print(explain_top(scores, firings, top_n=3))

Profile: Morgan (exploring options)
  Technical: python(2)
  Non-technical: communication(3)
  Interests: ai
  Work style: collaborative
  Short-term goals: explore what kinds of work I might enjoy
  Constraints: no_phd

TOP RECOMMENDATIONS

#1  Management Consulting   (score: +2.00)
    Why this ranks high:
      +2.00  Collaborative preference suits cross-functional roles.
             - work style: collaborative

#2  Technical Program Management   (score: +2.00)
    Why this ranks high:
      +2.00  Collaborative preference suits cross-functional roles.
             - work style: collaborative

#3  Product Management   (score: +1.50)
    Why this ranks high:
      +1.50  Collaborative preference suits cross-functional roles.
             - work style: collaborative


In [16]:
print(alternatives(profile, scores, firings))

ALTERNATIVE OPTIONS
  CLOSE CALL: Management Consulting (+2.00) and Technical Program Management (+2.00) are very close.
     Consider: Tech-strategy consulting / transformation lead
  WEAK MATCH: even the top option (Management Consulting) scores only +2.00.
     The profile may need more skill or experience signal before any of these careers becomes a strong fit. Use the counterfactuals below to see which additions would most change the picture.


In [17]:
print(counterfactuals(profile, scores))

COUNTERFACTUALS — what would change the recommendation?
  If you raised python to 4/5, top recommendation would shift to Software Engineering.
  If you raised machine learning to 4/5, top recommendation would shift to Machine Learning Engineering.
  If you raised statistics to 4/5, top recommendation would shift to Data Science.
  If you raised system design to 4/5, top recommendation would shift to DevOps / SRE.
  If you raised security to 4/5, top recommendation would shift to Cybersecurity.


**What to notice.** The system explicitly flags this as a *weak match* rather
than recommending the highest-scoring option as if it were a confident pick.
The counterfactuals here become *guidance* — concrete skills the user could
develop to make any career a stronger fit.

## 7. Tradeoff inspection

Some rules push one career up while pushing another down. The tradeoff inspector
surfaces these — useful for understanding *why* a particular profile constrains
the option space.

In [18]:
profile = research_candidate()
scores, firings = score(profile)
print(detect_tradeoffs(firings))

TRADEOFFS — rules that pull in opposing directions
  No active rules create opposing pressures on careers.


## All sample profiles at a glance

Beyond the five profiles walked through above, the project ships with
four more (cybersecurity, founder, quant, designer) to demonstrate that
the rule base generalizes. The cell below scores all nine and prints
each one's top-3 ranked careers in a compact table.


In [19]:
from sample_profiles import PROFILES

for name, fn in PROFILES.items():
    p = fn()
    s, _ = score(p)
    top3 = ranked(s)[:3]
    top_str = '  |  '.join(f'{cid} ({sc:+.1f})' for cid, sc in top3)
    print(f'{name:14s}  →  {top_str}')


swe             →  ml_engineering (+11.3)  |  entrepreneurship (+10.5)  |  software_engineering (+10.0)
research        →  research_academic (+19.1)  |  research_industry (+13.9)  |  data_science (+10.5)
consulting      →  product_management (+16.2)  |  consulting (+14.3)  |  entrepreneurship (+9.6)
conflicted      →  product_management (+15.3)  |  consulting (+10.6)  |  entrepreneurship (+10.0)
early_career    →  consulting (+2.0)  |  technical_program_management (+2.0)  |  product_management (+1.5)
cybersecurity   →  cybersecurity (+10.8)  |  software_engineering (+8.2)  |  devops_sre (+6.4)
founder         →  entrepreneurship (+25.5)  |  product_management (+18.3)  |  consulting (+15.2)
quant           →  finance_quant (+15.2)  |  research_academic (+12.2)  |  research_industry (+10.6)
designer        →  ux_design (+11.0)  |  product_management (+9.9)  |  software_engineering (+6.6)


## Build your own profile (interactive)

The cell below provides a live form: drag sliders to set skill levels,
click items in the multi-selects to add interests / work styles /
constraints, type free-text goals, then click **Get recommendations**.
The top 3 careers, alternatives, and counterfactuals update in place.

*Requires `ipywidgets` (bundled with most Jupyter installs). The
non-interactive equivalent — building a `UserProfile` directly — is
shown in the next cell as a fallback.*


In [20]:
import ipywidgets as widgets
from IPython.display import display, clear_output

skill_widgets = {
    s: widgets.IntSlider(value=v, min=0, max=5, description=s,
                         style={'description_width': '160px'},
                         layout=widgets.Layout(width='400px'))
    for s, v in [
        ('python', 4), ('machine_learning', 3), ('math', 3),
        ('statistics', 3), ('system_design', 2), ('finance', 0),
        ('security', 0), ('design_visual', 0),
    ]
}
ns_widgets = {
    s: widgets.IntSlider(value=v, min=0, max=5, description=s,
                         style={'description_width': '160px'},
                         layout=widgets.Layout(width='400px'))
    for s, v in [
        ('communication', 3), ('writing', 3),
        ('leadership', 2), ('public_speaking', 0),
    ]
}

interests = widgets.SelectMultiple(
    options=['ai', 'building', 'products', 'strategy', 'business',
             'research', 'theory', 'finance', 'markets', 'users',
             'security', 'open_source', 'infrastructure', 'llms',
             'healthcare', 'climate', 'education', 'gaming'],
    value=['ai', 'building'], rows=8, description='Interests',
    style={'description_width': '120px'},
)
work_style = widgets.SelectMultiple(
    options=['collaborative', 'independent', 'fast_paced', 'ambiguous',
             'strategic', 'hands_on', 'structured', 'ic_focused'],
    value=['collaborative', 'hands_on'], rows=6, description='Work style',
    style={'description_width': '120px'},
)
constraints = widgets.SelectMultiple(
    options=['no_phd', 'needs_visa_sponsor', 'no_relocation',
             'risk_averse', 'no_coding'],
    value=[], rows=5, description='Constraints',
    style={'description_width': '120px'},
)
goals = widgets.Textarea(
    value='find a role with technical depth and product impact',
    description='Goals (free text)', rows=2,
    style={'description_width': '160px'},
    layout=widgets.Layout(width='600px'),
)

out = widgets.Output()
button = widgets.Button(description='Get recommendations',
                        button_style='primary')

def on_click(_):
    with out:
        clear_output()
        profile = UserProfile(
            name='Interactive user',
            technical_skills={k: w.value for k, w in skill_widgets.items() if w.value > 0},
            non_technical_skills={k: w.value for k, w in ns_widgets.items() if w.value > 0},
            interests=list(interests.value),
            work_style=list(work_style.value),
            constraints=list(constraints.value),
            short_term_goals=[goals.value] if goals.value.strip() else [],
        )
        s, f = score(profile)
        print(profile.summary())
        print()
        print(explain_top(s, f, top_n=3, reasons_per_career=3))
        print()
        print(alternatives(profile, s, f))
        print()
        print(counterfactuals(profile, s))

button.on_click(on_click)

display(widgets.HBox([
    widgets.VBox([widgets.HTML('<b>Technical skills (0-5)</b>')] + list(skill_widgets.values())),
    widgets.VBox([widgets.HTML('<b>Non-technical skills (0-5)</b>')] + list(ns_widgets.values())),
]))
display(widgets.HBox([interests, work_style, constraints]))
display(goals)
display(button)
display(out)
on_click(None)  # initial run


Textarea(value='find a role with technical depth and product impact', description='Goals (free text)', layout=…

Button(button_style='primary', description='Get recommendations', style=ButtonStyle())

Output()

## 9. How the system explains its decisions

Every score in this system is fully traceable back to:

1. **A specific rule** in `advisor/rules.py` (with a human-readable description).
2. **A condition** that fired against the user's profile.
3. **The exact profile attributes** (skills, interests, work style, goals,
   constraints) that satisfied that condition — surfaced as `evidence` strings.
4. **A signed numeric contribution** to one or more career scores.

This is the opposite of a neural-network classifier: there is no opaque
weight matrix, no embedding, no black box. If the user disagrees with a
recommendation, they can read the trace and identify which specific rule
they disagree with — and the system tells them what would have to change
for the recommendation to flip.

The four explanation modes serve four distinct user questions:

- *"Why did you recommend this?"* → **`explain_top`**
- *"Why this one over that one?"* → **`head_to_head`**
- *"What would change your mind?"* → **`counterfactuals`**
- *"What's pulling against itself in my profile?"* → **`detect_tradeoffs`**

And `alternatives` adds a fifth: *"What other options should I consider?"* —
covering hybrid roles, dark horses, and weak-match warnings.